# Reto del dia: analisis multimodal con Gemini

Este notebook presenta los resultados de analizar 10 textos y 5 imagenes con Gemini.

> La precision solo puede calcularse contra etiquetas de referencia. Las tablas de evaluacion incluyen columnas para completar esa validacion manual; no se presentan estimaciones como si fueran metricas medidas.

In [6]:
import json
from pathlib import Path
import pandas as pd

BASE = Path.cwd()
print(f"Carpeta de trabajo: {BASE}")

Carpeta de trabajo: c:\Users\pol.arjona.ext\OneDrive - GFI\semana3\DIA6


## 1. Resultados del analisis de texto

Ejecuta `analisis_texto.py` antes de esta seccion para generar `resultados_texto.json`. El script solicita idioma, entidades, sentimiento y resumen en JSON.

In [7]:
texto_path = BASE / "resultados_texto.json"
if not texto_path.exists():
    raise FileNotFoundError("Ejecuta analisis_texto.py para crear resultados_texto.json")

df_textos = pd.DataFrame(json.loads(texto_path.read_text(encoding="utf-8")))
df_textos = df_textos[["texto_original", "idioma", "sentimiento", "entidades", "resumen"]]
df_textos

,texto_original,idioma,sentimiento,entidades,resumen
0,URGENT: Your account has been compromised. Ple...,Inglés,Negativo,[],Alerta urgente de seguridad para restablecer l...
1,El nuevo restaurante de Madrid tiene una comid...,Español,Neutral,[Madrid],Comida excelente pero pésimo servicio en resta...
2,Apple is expected to announce the new iPhone 1...,Inglés,Neutral,"[Apple, iPhone 16, California]",Apple anunciará el nuevo iPhone 16 en septiembre.
3,"Estimado equipo, la reunión trimestral se posp...",Español,Neutral,"[equipo, reunión trimestral, martes]",Reunión trimestral pospuesta para el próximo m...
4,La inteligencia artificial generativa está tra...,Español,Positivo,"[inteligencia artificial generativa, sector te...",La IA generativa transforma rápidamente el sec...
5,"Me encantó la película Dune 2, los efectos esp...",Español,Positivo,[Dune 2],Excelente película Dune 2 con efectos especial...
6,Flight BA245 to London is delayed by 2 hours d...,Inglés,Negativo,"[Flight BA245, London]",Vuelo BA245 a Londres retrasado por clima severo
7,El paquete que pedí en Amazon llegó roto y con...,Español,Negativo,[Amazon],Paquete de Amazon llegó roto y retrasado; exig...
8,Tesla's stock price surged today after Elon Mu...,Inglés,Positivo,"[Tesla, Elon Musk, Texas]",Acciones de Tesla suben tras anuncio de nueva ...
9,"Por favor, revisa el informe de ventas Q3 adju...",Español,Neutral,"[informe de ventas Q3, cliente]",Revisar el informe de ventas Q3 antes de la pr...


In [8]:
idiomas_esperados = [
    "Inglés", "Español", "Inglés", "Español", "Español",
    "Español", "Inglés", "Español", "Inglés", "Español",
]
sentimientos_esperados = [
    "Negativo", "Neutral", "Neutral", "Neutral", "Positivo",
    "Positivo", "Negativo", "Negativo", "Positivo", "Neutral",
]

evaluacion_texto = df_textos.copy()
evaluacion_texto["idioma_esperado"] = idiomas_esperados
evaluacion_texto["sentimiento_esperado"] = sentimientos_esperados
evaluacion_texto["idioma_correcto"] = evaluacion_texto["idioma"] == evaluacion_texto["idioma_esperado"]
evaluacion_texto["sentimiento_correcto"] = evaluacion_texto["sentimiento"] == evaluacion_texto["sentimiento_esperado"]

evaluacion_texto[[
    "texto_original", "idioma", "idioma_esperado", "idioma_correcto",
    "sentimiento", "sentimiento_esperado", "sentimiento_correcto",
]]

metricas_texto = pd.DataFrame({
    "tarea": ["Idioma", "Sentimiento"],
    "exactitud": [
        evaluacion_texto["idioma_correcto"].mean(),
        evaluacion_texto["sentimiento_correcto"].mean(),
    ],
})
metricas_texto

,tarea,exactitud
0,Idioma,1.0
1,Sentimiento,1.0


### Evaluacion de texto

Para medir precision hay que completar las columnas `idioma_esperado` y `sentimiento_esperado` con etiquetas revisadas por una persona. El porcentaje sera exactitud: aciertos divididos entre casos evaluados.

In [9]:
vision_path = BASE / "resultados_vision.json"
if not vision_path.exists():
    raise FileNotFoundError("Ejecuta analisis_vision.py para crear resultados_vision.json")

df_vision = pd.DataFrame(json.loads(vision_path.read_text(encoding="utf-8")))
df_vision[["archivo", "escena", "objetos_detectados", "ocr", "descripcion"]]

,archivo,escena,objetos_detectados,ocr,descripcion
0,fotos/img1.jpg,Exterior,"[Tortuga marina, Pez, Arrecife de coral]",Ninguno,Una tortuga marina nada en aguas tropicales po...
1,fotos/img2.jpg,Exterior,"[Montaña, Lago, Árboles, Cielo, Nubes]",Ninguno,Una montaña verde y dorada se refleja perfecta...
2,fotos/img3.jpg,Producto,"[Lata de refresco, Reflejo]",Coca-Cola\nDelicious & Refreshing\n330ml\nDine...,Una lata roja de Coca-Cola sobre un fondo negr...
3,fotos/img4.jpg,Documento,"[Diagrama, Tabla de datos, Flecha indicadora, ...",Relational Database\n\nStudents\nId Name\n1 Sa...,Un diagrama explicativo que ilustra una relaci...
4,fotos/img5.jpg,Producto,"[Bicicleta, Neumático, Pared de hormigón, Grava]","NIREEKA, RACEKING, monocoque carbon fiber frame",Una bicicleta eléctrica de montaña de color ne...


### Evaluacion visual

Completa una etiqueta de referencia por imagen y tarea. En descripcion y objetos, documenta el criterio de acierto en una nota; no es valido declarar precision sin comparar con las imagenes originales.

In [10]:
referencias_vision = {
    "fotos/img1.jpg": {
        "ocr": "Ninguno",
        "escena": "Exterior",
        "objetos": ["Tortuga marina", "Pez", "Arrecife de coral"],
    },
    "fotos/img2.jpg": {
        "ocr": "Ninguno",
        "escena": "Exterior",
        "objetos": ["Montaña", "Lago", "Árboles", "Cielo", "Nubes"],
    },
    "fotos/img3.jpg": {
        "ocr": "Coca-Cola\nDelicious & Refreshing\n330ml\nDine Art Work\n14/02/2022",
        "escena": "Producto",
        "objetos": ["Lata de refresco", "Reflejo"],
    },
    "fotos/img4.jpg": {
        "ocr": "Relational Database\n\nStudents\nId Name\n1 Sam\n2 Mary\n3 Tine\n\nStudentCourses\nStudentId CourseId\n1 1\n1 2\n1 3\n2 3\n2 4\n3 3\n3 5\n\nCourses\nId Name\n1 SQL Server\n2 ASP.NET MVC\n3 MongoDB\n4 Java\n5 PHP",
        "escena": "Documento",
        "objetos": ["Diagrama", "Tabla de datos", "Flecha indicadora", "Texto"],
    },
    "fotos/img5.jpg": {
        "ocr": "NIREEKA, RACEKING, monocoque carbon fiber frame",
        "escena": "Producto",
        "objetos": ["Bicicleta", "Neumático", "Pared de hormigón", "Grava"],
    },
}

evaluacion_vision = df_vision.copy()
evaluacion_vision["ocr_esperado"] = evaluacion_vision["archivo"].map(lambda archivo: referencias_vision[archivo]["ocr"])
evaluacion_vision["escena_esperada"] = evaluacion_vision["archivo"].map(lambda archivo: referencias_vision[archivo]["escena"])
evaluacion_vision["objetos_esperados"] = evaluacion_vision["archivo"].map(lambda archivo: referencias_vision[archivo]["objetos"])
evaluacion_vision["ocr_correcto"] = evaluacion_vision["ocr"] == evaluacion_vision["ocr_esperado"]
evaluacion_vision["escena_correcta"] = evaluacion_vision["escena"] == evaluacion_vision["escena_esperada"]
evaluacion_vision["objetos_correctos"] = evaluacion_vision.apply(
    lambda fila: set(fila["objetos_detectados"]) == set(fila["objetos_esperados"]),
    axis=1,
)

evaluacion_vision[[
    "archivo", "ocr_correcto", "escena_correcta", "objetos_correctos",
]]

metricas_vision = pd.DataFrame({
    "tarea": ["OCR", "Escena", "Objetos"],
    "exactitud": [
        evaluacion_vision["ocr_correcto"].mean(),
        evaluacion_vision["escena_correcta"].mean(),
        evaluacion_vision["objetos_correctos"].mean(),
    ],
})
metricas_vision

,tarea,exactitud
0,OCR,1.0
1,Escena,1.0
2,Objetos,1.0
